# 01-01 Historical Load

Downloads minute OHLCV bars from the **Massive API** for every ticker in the master universe that does not yet have bars in S3 staging. Writes one Parquet per ticker to `minute_data_staging/<TICKER>.parquet`.

The free tier rate-limits to ~5 calls/minute, so each ticker sleeps before the next request; a wall-clock deadline guards against session timeouts.

In [ ]:
# ============================================================================
# SETUP -- installs, imports, config (env vars / config.json -- never hardcoded)
# ============================================================================

# --- Install packages (no-op if already present) --------------------------
# !pip install -q -U massive duckdb tqdm
import os
import json
import sys
import time
from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import boto3
import duckdb
from massive import RESTClient
from tqdm.notebook import tqdm
# --- Configuration --------------------------------------------------------
# Secrets resolve in priority order:
#   1. Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY,
#      AWS_REGION, S3_BUCKET, MASSIVE_API_KEY, ...)
#   2. config.json in the current directory (see config.example.json)
#   3. Built-in defaults (non-secret values only)
# On Kaggle: set secrets via notebook settings (Add-ons -> Secrets), which
# are injected as environment variables.
CONFIG_FILE = "config.json"


def get_secret(name, default=""):
    val = os.environ.get(name)
    if val:
        return val
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            if name in data:
                return str(data[name])
        except (OSError, ValueError):
            pass
    return default


class Config:
    def __init__(self):
        self.aws_access_key_id = ""
        self.aws_secret_access_key = ""
        self.aws_region = "us-east-1"
        self.s3_bucket = "market-data-zw"
        self.massive_api_key = ""

        # Paths (S3 keys under the bucket)
        self.types_prefix = "parquet_data/types"
        self.tickers_prefix = "parquet_data/summary/tickers"
        self.ticker_details_prefix = "parquet_data/summary/ticker_yahoo_details"
        self.minute_staging_prefix = "parquet_data/minute_data_staging"
        self.minute_final_prefix = "parquet_data/minute_data_final"
        self.minute_summary_prefix = "parquet_data/summary/minute_summary"
        self.daily_volume_prefix = "parquet_data/summary/daily_volume"
        self.correlation_prefix = "parquet_data/strategies/correlation"
        self.backtest_prefix = "parquet_data/backtest"
        self.backtest_metrics_prefix = "parquet_data/analysis/backtest_metrics"

        # Spark
        self.spark_executor_memory = "24g"
        self.spark_executor_cores = 4
        self.spark_driver_memory = "24g"
        self.spark_tmp = "/tmp/spark"


def load_config():
    cfg = Config()
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            for key, value in data.items():
                if hasattr(cfg, key):
                    setattr(cfg, key, value)
        except (OSError, ValueError) as e:
            print(f"[config] WARNING: could not load {CONFIG_FILE}: {e}")

    env_map = {
        "AWS_ACCESS_KEY_ID": "aws_access_key_id",
        "AWS_SECRET_ACCESS_KEY": "aws_secret_access_key",
        "AWS_REGION": "aws_region",
        "S3_BUCKET": "s3_bucket",
        "MASSIVE_API_KEY": "massive_api_key",
        "SPARK_DRIVER_MEMORY": "spark_driver_memory",
        "SPARK_EXECUTOR_MEMORY": "spark_executor_memory",
        "SPARK_EXECUTOR_CORES": "spark_executor_cores",
    }
    for env_name, attr in env_map.items():
        val = os.environ.get(env_name)
        if val:
            if attr == "spark_executor_cores":
                val = int(val)
            setattr(cfg, attr, val)
    return cfg
# --- S3 helpers -----------------------------------------------------------
def s3_client(cfg):
    from botocore.config import Config as BotocoreConfig
    config = BotocoreConfig(retries={"max_attempts": 5, "mode": "adaptive"},
                            connect_timeout=30, read_timeout=60)
    return boto3.client("s3",
                        aws_access_key_id=cfg.aws_access_key_id,
                        aws_secret_access_key=cfg.aws_secret_access_key,
                        region_name=cfg.aws_region,
                        config=config)


def upload_parquet(df, s3, bucket, key, compression="snappy"):
    buf = BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow", compression=compression,
                  coerce_timestamps="ms", allow_truncated_timestamps=True)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())


def download_parquet(s3, bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))


def list_s3_keys(s3, bucket, prefix):
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys


def tickers_from_prefix(s3, bucket, prefix):
    """Ticker symbols from `<prefix>/<TICKER>.parquet` object keys."""
    return [k.split("/")[-1][:-len(".parquet")] for k in list_s3_keys(s3, bucket, prefix)
            if k.endswith(".parquet")]


def duckdb_s3_connect(cfg):
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"SET s3_access_key_id='{cfg.aws_access_key_id}';")
    con.execute(f"SET s3_secret_access_key='{cfg.aws_secret_access_key}';")
    con.execute(f"SET s3_region='{cfg.aws_region}';")
    return con


def spark_session(cfg):
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("MarketDataPlatform")
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1")
        .config("spark.executor.memory", cfg.spark_executor_memory)
        .config("spark.executor.cores", str(cfg.spark_executor_cores))
        .config("spark.driver.memory", cfg.spark_driver_memory)
        .config("spark.hadoop.fs.s3a.access.key", cfg.aws_access_key_id)
        .config("spark.hadoop.fs.s3a.secret.key", cfg.aws_secret_access_key)
        .config("spark.hadoop.fs.s3a.endpoint", f"s3.{cfg.aws_region}.amazonaws.com")
        .config("spark.local.dir", cfg.spark_tmp)
        .config("spark.hadoop.tmp.dir", cfg.spark_tmp)
        .config("spark.sql.warehouse.dir", f"{cfg.spark_tmp}/warehouse")
        .getOrCreate()
    )
    spark.conf.set("spark.hadoop.fs.s3a.committer.name", "directory")
    spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    spark.conf.set("spark.hadoop.fs.s3a.committer.staging.conflict-mode", "append")
    spark.conf.set("spark.sql.debug.maxToStringFields", "100")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    spark.conf.set("spark.sql.ansi.enabled", "false")
    spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
    spark.conf.set("spark.sql.parquet.mergeSchema", "true")
    spark.sparkContext.setLogLevel("ERROR")
    return spark

# --- Instantiate config + clients -----------------------------
cfg = load_config()
s3 = s3_client(cfg)
print("Setup complete")
print(f"Bucket: {cfg.s3_bucket} | Region: {cfg.aws_region}")


In [ ]:
# ============================================================================
# Session guard -- stop gracefully before the 12h Kaggle session limit
# ============================================================================

DEADLINE = time.time() + 11 * 60 * 60  # 11-hour guard

assert cfg.massive_api_key, "MASSIVE_API_KEY not set (env var or config.json)"
client = RESTClient(api_key=cfg.massive_api_key, retries=10)  # auto-retry on 429
print("Setup complete")

In [ ]:
# ============================================================================
# Tickers that already have minute bars = object KEYS under minute_data_staging/
# ============================================================================

s3_tickers_list = tickers_from_prefix(s3, cfg.s3_bucket, f"{cfg.minute_staging_prefix}/")
print(f"{len(s3_tickers_list):,} tickers already have minute bars")

In [ ]:
# ============================================================================
# Master universe (single file from 00-01)
# ============================================================================

con = duckdb_s3_connect(cfg)
q = f"SELECT DISTINCT ticker FROM read_parquet('s3://{cfg.s3_bucket}/{cfg.tickers_prefix}/tickers.parquet')"
tickers_list = [r[0] for r in con.execute(q).fetchall()]
print(f"{len(tickers_list):,} tickers in the master universe")

In [ ]:
# ============================================================================
# Tickers in the master universe but without minute bars yet
# ============================================================================

import random

missing_tickers = sorted(set(tickers_list) - set(s3_tickers_list))
random.shuffle(missing_tickers)
print(f"Master: {len(tickers_list):,} | Have bars: {len(s3_tickers_list):,} | Missing: {len(missing_tickers):,}")

In [ ]:
# ============================================================================
# Fetch minute bars for one ticker (auto-paginated, 50k bars per call)
# ============================================================================

def fetch_ticker_bars(ticker, from_date, to_date):
    rows = []
    for a in client.list_aggs(ticker=ticker, multiplier=1, timespan="minute",
                              from_=from_date, to=to_date, limit=50000):
        rows.append({
            "symbol": ticker,
            "date": a.timestamp,   # epoch ms
            "open": a.open,
            "high": a.high,
            "low": a.low,
            "close": a.close,
            "volume": a.volume,
            "vwap": a.vwap,
            "trades": a.transactions,
        })
    return pd.DataFrame(rows)

In [ ]:
# ============================================================================
# Gap backfill window (adjust to your collection period)
# ============================================================================

FROM_DATE = "2026-01-01"
TO_DATE   = "2026-02-28"
print(f"Fetching minute bars from {FROM_DATE} to {TO_DATE}")

stats = {"success": 0, "empty_response": 0, "errors": {}, "skipped_tickers": []}

for ticker in tqdm(missing_tickers, desc="Downloading minute bars", unit="ticker"):
    if time.time() > DEADLINE:
        print("Time limit reached, stopping.")
        break

    try:
        df = fetch_ticker_bars(ticker, FROM_DATE, TO_DATE)
        if df.empty:
            stats["empty_response"] += 1
            stats["skipped_tickers"].append((ticker, "empty response"))
            tqdm.write(f"  {ticker}: EMPTY - no bars returned")
            continue

        upload_parquet(df, s3, cfg.s3_bucket,
                       f"{cfg.minute_staging_prefix}/{ticker}.parquet", compression="zstd")
        stats["success"] += 1
        tqdm.write(f"  {ticker}: OK ({len(df)} bars)")
    except Exception as e:
        err_type = type(e).__name__
        stats["errors"][err_type] = stats["errors"].get(err_type, 0) + 1
        stats["skipped_tickers"].append((ticker, f"{err_type}: {e}"))
        tqdm.write(f"  {ticker}: ERROR - {err_type}: {e}")

    time.sleep(12)  # ~5 calls/min free-tier rate limit

# --- Summary -----------------------------------------------------------------
print("\n=== DOWNLOAD SUMMARY ===")
print(f"Successful:        {stats['success']}")
print(f"Empty responses:   {stats['empty_response']}")
for err_type, count in stats["errors"].items():
    print(f"  {err_type}: {count}")
print(f"Total skipped:     {len(stats['skipped_tickers'])}")
print(f"Downloaded minute bars -> {cfg.minute_staging_prefix}/")